# 25 carrier unix

Question: does `graphtp-homa` over Unix Socket, served by the GraphTP listener of `aosabi-core` as
the card `test-graphtp`, keep the rules of streams, restarts and at most once through a real socket?

The side under test is the operation `serve` of `aosabi-devtool graphtp`, the card `test-graphtp` of
the GraphTP track: the listener of `aosabi-core` under its feature `homa` over the server of
`graphtp-homa`, one Homa endpoint of the server role and one listener per socket, on a
`SOCK_SEQPACKET` socket and on the stream socket of the prefixed framing that `AOSABI_UNIX_STREAM`
names, with the service of the harness behind the listener. The reference side is the client of
`tests/harness/homa/graphtp.py` over the Python Homa engine `tests/harness/homa/client.py`, both
written apart from the crates, and this notebook, which knows every run the service counted and
every entry each stream was asked to carry.

The exit number of the server rows, `docs/graphtp-in-aosabi.md` section 9, on both forms of the
socket: the restart cases of `docs/graphtp.md` section 2.7 with 0 double runs and every unknown
outcome reported, one parked PULL serving ten streams, 409.1 on a second PULL, batches applied by
`seq` under injected reordering with 0 lost events, CANCEL of one handle, of the peer's last
handle and of handle 0 each answering the parked PULL with 404.1 first, a restarted client taking
the count of `seq` from its CANCEL of handle 0, 429.1 at the bound, and a link down ending every
stream of the peer and setting its count back to 0, the reset judged on one listener whose link
goes down and up again, since a new connection holds a listener of its own. The sizes of the
exchanges of `docs/graphtp.md` section 6.2 that the probe of batch GraphTP-B6 printed are held as
well.

The plan places them in G3 with the other server rows, while section 9 gives them no exit number, so
they are a leftover of G3 that the record of the GraphTP track moves to G6 by default, and the plan
gains them in the row of G6 at merge. Each prints not run below, which is never a pass.

Specification: `docs/graphtp-in-aosabi.md` sections 3 and 9, and `docs/graphtp.md` sections 2.4, 2.7
and 6.2.

In [ ]:
# Papermill overrides this cell.
# The tree under comparison.
aosabi_root = "."

# The forms of the socket every section runs on, comma separated.
forms = "seqpacket,stream"

# The seed of the reordering and of the loss hooks.
seed = 20260928

# The permutations of the stream answers the reordering applies.
permutations = 200

# The OPERATIONs of each loss run, and the loss rates of the hooks per million
# packets, with the duplication beside them.
loss_operations = 200
loss_rates = "10000,50000"
duplicate_ppm = 10000

# The per command timeout, in seconds.
timeout_s = 600

# The pinned toolchain the crates are built with.
rust_version = "1.95.0"

## Setup

The cell prints the toolchain and runs the gates the notebook rests on, every cargo call with
`--locked`: the build of the devtool, the lints of `graphql-over-graphtp` and of `aosabi-core` with
its feature `homa` alone, which is the build of a card with no standard library, and the format of
the files of the listener and of the card. It then starts nothing, and each section starts its own
card, so the runs and the restarts one section counts belong to it alone.

Both ends run the timers of the Unix Socket profile shortened to a RESEND after 20 ms and a patient
request asked after 200 ms, since the 1,000 ms and 5 s of the profile would make each restart case
last seconds, and a peer is dead after 10 unanswered questions and 200 ms of silence.

In [ ]:
import json
import os
import random
import subprocess
import sys
import time
from pathlib import Path

for _base in [Path.cwd(), *Path.cwd().parents]:
    if (_base / "tests" / "harness" / "__init__.py").is_file():
        sys.path.insert(0, str(_base / "tests"))
        break

from harness import paths, run
from harness.homa import client as hc
from harness.homa import graphtp as g

P = paths(aosabi_root)
OUT = P.out / "25_carrier_unix"
SOCKS = OUT / "sock"
SOCKS.mkdir(parents=True, exist_ok=True)
DEVTOOL = P.target / "debug" / "aosabi-devtool"
CARGO_BIN = Path(os.environ.get("CARGO_HOME") or Path.home() / ".cargo") / "bin"
CARGO = CARGO_BIN / "cargo"
RUSTFMT = CARGO_BIN / "rustfmt"
FORMS = forms.split(",")

GATES = [
    ("cargo build --locked -p aosabi-devtool", [CARGO, "build", "--locked", "-p", "aosabi-devtool"]),
    ("cargo clippy --locked -p aosabi-core --no-default-features --features homa -- -D warnings",
     [CARGO, "clippy", "--locked", "-p", "aosabi-core", "--no-default-features", "--features", "homa",
      "--", "-D", "warnings"]),
    ("cargo clippy --locked -p graphql-over-graphtp -- -D warnings",
     [CARGO, "clippy", "--locked", "-p", "graphql-over-graphtp", "--", "-D", "warnings"]),
]
LISTENER_FILES = [*sorted((P.root / "crates" / "graphql-over-graphtp" / "src").glob("*.rs")),
                  P.root / "crates" / "aosabi-core" / "src" / "graphtp.rs",
                  *sorted((P.root / "crates" / "aosabi-core" / "src" / "graphtp").glob("*.rs"))]
CARD_FILES = [P.root / "crates" / "aosabi-devtool" / "src" / "graphtp" / "card.rs"]
for path in LISTENER_FILES + CARD_FILES:
    GATES.append(("rustfmt --check --edition 2024 " + P.rel(path), [RUSTFMT, "--check", "--edition", "2024", path]))
GATE_FAILURES = []
for label, argv in GATES:
    gate = run(argv, cwd=P.root, timeout_s=900, check=False)
    print("gate %-90s exit %d, %.1f s" % (label, gate.returncode, gate.duration_s))
    if gate.returncode != 0:
        GATE_FAILURES.append(label)
        print((gate.err or gate.out).strip()[-2000:])

reported_cargo = run([CARGO, "--version"], cwd=P.root, timeout_s=timeout_s).out.strip()
smoke = json.loads(run([DEVTOOL, "graphtp", "--smoke"], cwd=P.root, timeout_s=timeout_s).out)
print("repository root:", P.rel(P.root))
print("devtool        :", P.rel(DEVTOOL))
print("cargo          :", reported_cargo)
print("seam           :", json.dumps(smoke))
print("forms          :", ", ".join(FORMS))
assert not GATE_FAILURES, GATE_FAILURES
assert rust_version in reported_cargo, reported_cargo
assert smoke == {"subcommand": "graphtp", "implemented": True, "lands_at": "M12"}, smoke

PANICS = []
# The OPERATIONs a client keeps open at once, half the RPCs the card
# announces, so the engine never refuses one as Full.
WINDOW = 8
FAST = {"resend_after_ns": 20_000_000, "resend_interval_ns": 20_000_000, "need_ack_after_ns": 40_000_000,
        "patient_after_ns": 200_000_000, "rpc_timeout_ns": 200_000_000, "timeout_resends": 10}
MAX_MESSAGE = 1 << 20


def py_profile():
    """The Unix Socket profile of client.py with the timers of FAST."""
    return hc.unix_socket(MAX_MESSAGE).override(resend_after=0.02, resend_interval=0.02, need_ack_after=0.04,
                                                patient_after=0.2, rpc_timeout=0.2, timeout_resends=10)


def serve(name, connections=1, card=None, config=None, max_time_ms=120_000):
    """Start the card test-graphtp in the background and wait for both sockets."""
    sock = SOCKS / (name + ".homa")
    stream = SOCKS / (name + ".stream")
    for path in (sock, stream):
        if path.exists():
            path.unlink()
    request = {"id": 0, "op": "serve", "path": str(sock), "profile": {"name": "unix_socket", "max_message": MAX_MESSAGE},
               "overrides": FAST, "limits": {"max_message": MAX_MESSAGE}, "connections": connections,
               "max_time_ms": max_time_ms}
    if card:
        request["card"] = card
    if config:
        request["config"] = config
    lines = OUT / (name + ".ndjson")
    lines.write_text(json.dumps(request) + "\n", encoding="utf-8")
    env = dict(os.environ, AOSABI_UNIX_STREAM=str(stream))
    proc = subprocess.Popen([str(DEVTOOL), "graphtp", "--requests", str(lines)], cwd=P.root, env=env,
                            stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    hc.wait_for(sock)
    hc.wait_for(stream)
    return {"proc": proc, "name": name, "seqpacket": sock, "stream": stream}


def served(card):
    """Wait for a card to end and return its answer."""
    out, err = card["proc"].communicate(timeout=timeout_s)
    answers = [json.loads(line) for line in out.splitlines()]
    if card["proc"].returncode != 0 or len(answers) != 1 or "fatal" in answers[0] or answers[0]["stopped"] != "done":
        PANICS.append("%s: exit %d, %s, %s" % (card["name"], card["proc"].returncode, out.strip()[-400:],
                                               err.strip()[-400:]))
        return answers[0] if answers else {}
    return answers[0]


def client(card, form, hook=None, first_id=0x30000002, count_known=True):
    """A GraphTP client of graphtp.py over a new engine of client.py on one form of the card."""
    return g.connect(form, card[form], profile=py_profile(), max_message=MAX_MESSAGE, hook=hook,
                     first_id=first_id, count_known=count_known)


def close(c):
    """Close the client's socket, which the card reads as the link going down."""
    c.engine.t.sock.close()


def stream_content(tag, events=3, interval_ms=0, fail_after=None):
    """The content of an OPERATION whose answer is a stream of the service."""
    spec = {"events": events, "interval_ms": interval_ms, "tag": tag}
    if fail_after is not None:
        spec["fail_after"] = fail_after
    return g.harness_content({"stream": spec})


def next_ticks(stream):
    """The ticks and the tags of the next values a stream carried."""
    return [json.loads(value)["data"] for value in stream["next"]]


def pull_until_closed(c, limit=500):
    """PULL, one parked at a time, until the client holds no open stream."""
    batches = []
    while c.open_streams() and len(batches) < limit:
        batches.append(c.run(c.pull(), timeout=30))
    return batches

## 1. The sizes of the exchanges of section 6.2

The calls of `docs/graphtp.md` section 6.2 run against the card: `subscription{tick}`, which the
service answers with a stream of two results 100 ms apart, then two PULLs, a CANCEL of a second
stream, a PULL of a peer with no stream, and `mutation{pause}`. The cell prints the bytes of each
message as it crossed the socket, the request as `graphtp.py` wrote it and the answer as the
listener wrote it, and holds each to the table of that section, which the probe of batch GraphTP-B6
printed for the server of `graphtp-homa` alone.

In [ ]:
SIZES = {}
EXPECTED_SIZES = {"OPERATION subscription{tick}": 33, "200.3 with its head": 34, "PULL": 3,
                  "200.0 with one event": 62, "200.0 with the last event and complete": 91,
                  "CANCEL": 5, "200.0 with one cancelled": 52, "404.1, a refusal": 2,
                  "OPERATION mutation{pause}": 30, "200.0 {pause:true}": 25}
EXPECTED_BODIES = {
    "200.3 with its head": b'{"stream":1,"seq":0,"events":[]}',
    "200.0 with one event": b'{"seq":1,"events":[{"stream":1,"next":{"data":{"tick":1}}}]}',
    "200.0 with the last event and complete":
        b'{"seq":2,"events":[{"stream":1,"next":{"data":{"tick":2}}},{"stream":1,"complete":true}]}',
    "200.0 with one cancelled": b'{"seq":4,"events":[{"stream":2,"cancelled":true}]}',
}
for form in FORMS:
    card = serve("sizes-" + form)
    c = client(card, form)
    rows = {}
    content = b'{"query":"subscription{tick}"}'
    rows["OPERATION subscription{tick}"] = (g.write_operation(None, content), None)
    op = c.operation(content)
    c.wait([op])
    rows["200.3 with its head"] = (op.rpc.response, c.take(op))
    pull = c.pull()
    rows["PULL"] = (g.write_pull(), None)
    c.wait([pull])
    rows["200.0 with one event"] = (pull.rpc.response, c.take(pull))
    pull = c.pull()
    c.wait([pull])
    rows["200.0 with the last event and complete"] = (pull.rpc.response, c.take(pull))
    second = c.operation(stream_content("cancel-me", events=-1, interval_ms=10_000))
    c.wait([second])
    handle = c.take(second)["stream"]
    cancel = c.cancel(handle)
    rows["CANCEL"] = (g.write_cancel(handle), None)
    c.wait([cancel])
    rows["200.0 with one cancelled"] = (cancel.rpc.response, c.take(cancel))
    lone = c.pull()
    c.wait([lone])
    rows["404.1, a refusal"] = (lone.rpc.response, c.take(lone))
    content = b'{"query":"mutation{pause}"}'
    rows["OPERATION mutation{pause}"] = (g.write_operation(None, content), None)
    call = c.operation(content)
    c.wait([call])
    rows["200.0 {pause:true}"] = (call.rpc.response, c.take(call))
    close(c)
    answer = served(card)
    SIZES[form] = {label: len(message) for label, (message, _) in rows.items()}
    for label, (message, outcome) in rows.items():
        body = outcome.get("body") if outcome else None
        print("%-9s %-42s %4d bytes  %s" % (form, label, len(message), "" if body is None else body.decode()))
    for label, expected in EXPECTED_BODIES.items():
        assert rows[label][1]["body"] == expected, (form, label, rows[label][1])
    assert rows["404.1, a refusal"][1]["code"] == g.Code(404, 1), rows["404.1, a refusal"]

## 2. One parked PULL for ten streams

Ten OPERATIONs open ten streams of three results each, 30 ms apart, each tagged with its number. The
client then parks one PULL at a time until every stream is complete, and each batch carries the
ready entries of every stream of the peer. The cell counts the PULLs, the streams a batch carried at
most, and every result against the stream and the tick it was asked to carry.

In [ ]:
TEN = {}
for form in FORMS:
    card = serve("ten-" + form)
    c = client(card, form)
    ops = [c.operation(stream_content("t%d" % n, events=3, interval_ms=30)) for n in range(10)]
    c.wait(ops)
    heads = [c.take(op) for op in ops]
    batches = pull_until_closed(c)
    close(c)
    answer = served(card)
    per_stream = {}
    for head, n in zip(heads, range(10)):
        stream = c.streams[head["stream"]]
        per_stream["t%d" % n] = (stream["state"], next_ticks(stream))
    wrong = [tag for tag, (state, ticks) in per_stream.items()
             if state != "complete" or ticks != [{"tick": k, "tag": tag} for k in (1, 2, 3)]]
    widest = max(len({entry[1] for entry in batch["entries"]}) for batch in batches)
    TEN[form] = {"opened": sum(head["code"] == g.STREAM_OPENED for head in heads), "pulls": len(batches),
                 "widest_batch": widest, "results": sum(len(ticks) for _, ticks in per_stream.values()),
                 "complete": sum(state == "complete" for state, _ in per_stream.values()), "wrong": wrong,
                 "held": len(c.held), "dropped": c.dropped, "service": answer.get("service", {})}
    print(form, TEN[form])

## 3. A second PULL beside the parked one

A stream whose second result comes after 10 s has its first result taken by a PULL, then a PULL is
parked, and a second PULL beside it is refused with 409.1 while the first stays parked. The CANCEL
of the stream, the peer's one stream, then answers the parked PULL with 404.1 before its own batch,
which the order the client engine ended the two RPCs shows.

In [ ]:
PARKED = {}
for form in FORMS:
    card = serve("parked-" + form)
    c = client(card, form)
    handle = c.run(c.operation(stream_content("slow", events=-1, interval_ms=10_000)))["stream"]
    first = c.run(c.pull())
    parked = c.pull()
    deadline = time.monotonic() + 0.3
    while time.monotonic() < deadline:
        c.engine.poll(0.002)
    second = c.run(c.pull())
    still_parked = not parked.done
    cancel = c.cancel(handle)
    c.wait([cancel, parked])
    PARKED[form] = {"first": str(first["code"]), "second": str(second["code"]), "still_parked": still_parked,
                    "parked": str(c.take(parked)["code"]), "cancel": [e[0] for e in c.take(cancel)["entries"]],
                    "pull_first": parked.rpc.ended < cancel.rpc.ended}
    close(c)
    answer = served(card)
    PARKED[form]["returned"] = answer.get("service", {}).get("returned")
    print(form, PARKED[form])

## 4. Batches applied by `seq` under injected reordering

Ten streams of four results 15 ms apart are served to the end, and the client keeps every stream
answer it read, the ten heads and every batch, without applying them. A fresh client then applies
the same answers in `permutations` orders drawn with the seed, and each order must give every stream
its four results in order and its `complete`, hold no answer at the end and drop none, as the
application in the order of `seq` gives. The answers come from the listener as it wrote them, so the
reorder injected is the one Homa may cause between the answers of two RPCs.

In [ ]:
REORDER = {}
rng = random.Random(seed)
for form in FORMS:
    card = serve("reorder-" + form)
    c = client(card, form)
    ops = [c.operation(stream_content("r%d" % n, events=4, interval_ms=15)) for n in range(10)]
    c.wait(ops)
    answers = [c.take(op, apply=False) for op in ops]
    completes = 0
    while completes < 10:
        batch = c.run(c.pull(), timeout=30)
        answers.append(batch)
        completes += sum(entry[0] == "complete" for entry in batch["entries"])
    close(c)
    served(card)

    def applied(order):
        fresh = g.Client(None)
        for answer in order:
            fresh.accept(answer)
        return fresh

    reference = applied(sorted(answers, key=lambda answer: answer["seq"]))
    expect = {handle: (stream["state"], stream["next"]) for handle, stream in reference.streams.items()}
    lost = sum(4 - len(stream["next"]) for stream in reference.streams.values())
    differing = 0
    for _ in range(permutations):
        order = rng.sample(answers, len(answers))
        fresh = applied(order)
        got = {handle: (stream["state"], stream["next"]) for handle, stream in fresh.streams.items()}
        if got != expect or fresh.held or fresh.dropped or fresh.expected != len(answers):
            differing += 1
    REORDER[form] = {"answers": len(answers), "streams": len(expect), "permutations": permutations,
                     "differing": differing, "lost": lost,
                     "complete": sum(state == "complete" for state, _ in expect.values())}
    print(form, REORDER[form])

## 5. CANCEL of one handle, of the last handle and of handle 0

Three cases, each with a PULL parked when the CANCEL leaves. A CANCEL of one handle while the peer
holds a second stream leaves the parked PULL parked and answers one `cancelled` entry. A CANCEL of
the peer's last handle then answers the parked PULL with 404.1 before its own batch. A CANCEL of
handle 0 over three streams answers the parked PULL with 404.1 before its batch of three `cancelled`
entries. The service counts every source a CANCEL returned.

In [ ]:
CANCELS = {}


def park(c):
    """Park a PULL and let it reach the card."""
    pull = c.pull()
    deadline = time.monotonic() + 0.2
    while time.monotonic() < deadline:
        c.engine.poll(0.002)
    return pull


for form in FORMS:
    card = serve("cancels-" + form)
    c = client(card, form)
    handles = [c.run(c.operation(stream_content("c%d" % n, events=-1, interval_ms=10_000)))["stream"]
               for n in range(2)]
    c.run(c.pull())
    parked = park(c)
    one = c.cancel(handles[0])
    c.wait([one])
    kept = not parked.done
    one_entries = [tuple(entry) for entry in c.take(one)["entries"]]
    last = c.cancel(handles[1])
    c.wait([last, parked])
    last_case = {"parked": str(c.take(parked)["code"]), "pull_first": parked.rpc.ended < last.rpc.ended,
                 "entries": [tuple(entry) for entry in c.take(last)["entries"]]}
    three = [c.run(c.operation(stream_content("z%d" % n, events=-1, interval_ms=10_000)))["stream"]
             for n in range(3)]
    c.run(c.pull())
    parked = park(c)
    everything = c.cancel(0)
    c.wait([everything, parked])
    zero_case = {"parked": str(c.take(parked)["code"]), "pull_first": parked.rpc.ended < everything.rpc.ended,
                 "entries": sorted(tuple(entry) for entry in c.take(everything)["entries"])}
    close(c)
    answer = served(card)
    CANCELS[form] = {"one": {"kept_parked": kept, "entries": one_entries, "handle": handles[0]},
                     "last": dict(last_case, handle=handles[1]), "zero": dict(zero_case, handles=three),
                     "returned": answer.get("service", {}).get("returned")}
    print(form, CANCELS[form])

## 6. A restarted client takes its count from CANCEL of handle 0

A client opens two endless streams, takes a batch and parks a PULL, so the card counts `seq` for its
peer past 0. The client then restarts on the same socket: a new engine of `client.py` with other
identifiers and a client that does not know the count. By the start rule it sends a CANCEL of handle
0 and waits for its answer, which ends both streams, answers the old parked PULL with 404.1 and
carries the count the card kept, and the client expects one more next. A new stream of three results
then opens at that count and is applied at once, with 0 lost events.

In [ ]:
RESTARTED = {}
for form in FORMS:
    card = serve("restarted-" + form)
    c = client(card, form)
    for n in range(2):
        c.run(c.operation(stream_content("old%d" % n, events=-1, interval_ms=10_000)))
    c.run(c.pull())
    parked = park(c)
    counted = c.expected
    profile = py_profile()
    engine = hc.Engine(c.engine.t, profile, hc.limits(profile, MAX_MESSAGE), first_id=0x31000002)
    fresh = g.Client(engine, count_known=False)
    start = fresh.start()
    head = fresh.run(fresh.operation(stream_content("new", events=3, interval_ms=10)))
    held_after_head = len(fresh.held)
    pull_until_closed(fresh)
    new_stream = fresh.streams[head["stream"]]
    close(fresh)
    answer = served(card)
    RESTARTED[form] = {"count_of_the_old_run": counted, "cancel_seq": start["seq"],
                       "cancelled": sorted(entry[1] for entry in start["entries"]),
                       "expected_after_start": (start["seq"] + 1) % 2**32, "head_seq": head["seq"],
                       "held_after_head": held_after_head, "new_state": new_stream["state"],
                       "new_results": next_ticks(new_stream), "dropped": fresh.dropped,
                       "returned": answer.get("service", {}).get("returned")}
    print(form, RESTARTED[form])

## 7. At most once: the restart cases of section 2.7

Every OPERATION of this section asks the service to count a run of its own identifier, so a request
that runs twice shows as a count of 2. An OPERATION the caller does not mark reports `RPC_UNKNOWN`
in place of Homa's silent restart, and a marked one keeps the restart.

- **While the card holds the RPC.** The client sends `loss_operations` OPERATIONs, unmarked and
  marked in turn, through a hook of `client.py` that drops and duplicates packets both ways at each
  rate of `loss_rates`. Each ends answered, or unmarked and reported unknown where its request was
  lost whole, and none runs twice.
- **A restart of the card after the run.** An OPERATION whose run asks the service to rebuild the
  endpoint and the listener of the connection, and whose answer comes 300 ms later, so the card
  that answers the client's RESEND holds no such RPC. Unmarked, it reports 502.1 and ran once.
  Marked, Homa repeats it, the new card runs it again, and its answer counts 2, the repeat its
  caller allowed.
- **A restart of the card before the run.** An OPERATION whose run waits 400 ms, and a second one
  100 ms later whose run rebuilds the connection at once, so the first never runs on the card that
  took it. Unmarked, it reports 502.1 and never ran. Marked, Homa repeats it, and it runs once on
  the new card.
- **After a link down.** An OPERATION whose answer comes 500 ms later, after the client took its
  link down and closed the socket. It reports 502.2 and ran once.
- **A client silent past the dead peer rule.** An OPERATION runs and is answered while the client
  reads and sends nothing, the card declares the peer dead and drops the RPC, and the client, which
  never read the answer, meets a card that holds no such RPC. It reports an unknown outcome and ran
  once.
- **A parked PULL across a restart of the card.** The PULL is repeatable, so Homa repeats it, the
  new card refuses it with 404.1, the client loses its stream, sends a CANCEL of handle 0 by the
  start rule and takes the count 0 from its answer.

In [ ]:
AT_MOST_ONCE = {}
for form in FORMS:
    rows = {}
    for rate in [int(value) for value in loss_rates.split(",")]:
        card = serve("loss-%s-%d" % (form, rate))
        hook = hc.Loss(seed + rate, drop_ppm=rate, duplicate_ppm=duplicate_ppm)
        c = client(card, form, hook=hook)
        exchanges = []
        for n in range(loss_operations):
            while sum(not exchange.done for _, _, exchange in exchanges) >= WINDOW:
                c.engine.poll(0.002)
            marked = n % 2 == 1
            ident = "l%d-%d" % (rate, n)
            exchanges.append((ident, marked, c.operation(g.harness_content({"run": ident}), repeatable=marked)))
        c.wait([exchange for _, _, exchange in exchanges], timeout=120)
        outcomes = [(ident, marked, exchange.outcome()) for ident, marked, exchange in exchanges]
        close(c)
        runs = served(card).get("service", {}).get("runs", {})
        rows["loss %d ppm" % rate] = {
            "operations": len(outcomes),
            "answered": sum(outcome["answer"] == "result" for _, _, outcome in outcomes),
            "unknown_reported": sum(outcome["answer"] == "unknown" for _, _, outcome in outcomes),
            "unknown_marked": sum(outcome["answer"] == "unknown" and marked for _, marked, outcome in outcomes),
            "other": sum(outcome["answer"] not in ("result", "unknown") for _, _, outcome in outcomes),
            "double_runs": sum(count > 1 for count in runs.values()),
            "answer_counts_wrong": sum(1 for ident, _, outcome in outcomes if outcome["answer"] == "result"
                                       and json.loads(outcome["body"])["data"]["count"] != runs.get(ident)),
            "hook": dict(hook.counts),
        }
    for marked in (False, True):
        card = serve("restart-%s-%s" % (form, str(marked).lower()))
        c = client(card, form)
        ident = "restart-%s" % str(marked).lower()
        outcome = c.run(c.operation(g.harness_content({"run": ident, "restart": True, "delay_ms": 300}),
                                    repeatable=marked), timeout=30)
        close(c)
        answer = served(card)
        runs = answer.get("service", {}).get("runs", {})
        rows["restart after the run, %s" % ("marked" if marked else "unmarked")] = {
            "answer": outcome["answer"], "code": str(outcome["code"]),
            "count_in_answer": json.loads(outcome["body"])["data"]["count"] if outcome["answer"] == "result" else None,
            "runs": runs.get(ident), "restarts": sum(conn["restarts"] for conn in answer.get("connections", []))}
    for marked in (False, True):
        card = serve("before-%s-%s" % (form, str(marked).lower()))
        c = client(card, form)
        ident = "before-%s" % str(marked).lower()
        early = c.operation(g.harness_content({"run": ident, "wait_ms": 400}), repeatable=marked)
        deadline = time.monotonic() + 0.1
        while time.monotonic() < deadline:
            c.engine.poll(0.002)
        trigger = c.run(c.operation(g.harness_content({"run": ident + "-trigger", "restart": True})))
        c.wait([early], timeout=30)
        outcome = c.take(early)
        close(c)
        answer = served(card)
        runs = answer.get("service", {}).get("runs", {})
        rows["restart before the run, %s" % ("marked" if marked else "unmarked")] = {
            "answer": outcome["answer"], "code": str(outcome["code"]), "trigger": trigger["answer"],
            "count_in_answer": json.loads(outcome["body"])["data"]["count"] if outcome["answer"] == "result" else None,
            "runs": runs.get(ident, 0), "restarts": sum(conn["restarts"] for conn in answer.get("connections", []))}
    card = serve("linkdown-" + form)
    c = client(card, form)
    late = c.operation(g.harness_content({"run": "late", "delay_ms": 500}))
    deadline = time.monotonic() + 0.1
    while time.monotonic() < deadline:
        c.engine.poll(0.002)
    c.engine.link_down()
    close(c)
    outcome = late.outcome()
    answer = served(card)
    rows["link down"] = {"answer": outcome["answer"], "code": str(outcome.get("code")),
                         "runs": answer.get("service", {}).get("runs", {}).get("late")}
    card = serve("silent-" + form)
    c = client(card, form)
    quiet = c.operation(g.harness_content({"run": "quiet"}))
    deadline = time.monotonic() + 0.05
    while time.monotonic() < deadline and c.engine.stats["sent"] < 2:
        c.engine.poll(0.002)
    c.engine.silent = True
    time.sleep(1.0)
    c.engine.t.hook = lambda direction, packet: [] if direction == "in" else [packet]
    c.engine.silent = False
    for _ in range(5):
        c.engine.poll(0.01)
    c.engine.t.hook = None
    c.wait([quiet], timeout=10)
    outcome = quiet.outcome()
    close(c)
    answer = served(card)
    rows["silent past the dead peer rule"] = {
        "answer": outcome["answer"], "code": str(outcome.get("code")),
        "runs": answer.get("service", {}).get("runs", {}).get("quiet"),
        "peer_deaths": sum(conn["stats"]["peer_deaths"] for conn in answer.get("connections", []))}
    card = serve("pullrestart-" + form)
    c = client(card, form)
    handle = c.run(c.operation(stream_content("across", events=-1, interval_ms=10_000)))["stream"]
    c.run(c.pull())
    parked = park(c)
    trigger = c.run(c.operation(g.harness_content({"run": "trigger", "restart": True})))
    c.wait([parked], timeout=30)
    refused = c.take(parked)
    lost = c.streams[handle]["state"]
    start = c.start()
    close(c)
    answer = served(card)
    rows["parked PULL across a restart"] = {"trigger": trigger["answer"], "pull": str(refused["code"]),
                                            "stream": lost, "start_seq": start.get("seq"),
                                            "expected": c.expected,
                                            "restarts": sum(conn["restarts"] for conn in answer.get("connections", []))}
    AT_MOST_ONCE[form] = rows
    for label, row in rows.items():
        print("%-9s %-32s %s" % (form, label, row))

## 8. The stream bound and back pressure

The card `test-graphtp` lets one peer hold 16 streams and queues 8 results a stream between two
batches. The client opens 16 endless streams, the card refuses the 17th with 429.1, and after a
CANCEL of one a new one opens. A stream of 20 results with no time between them is read only while
its queue has room, so its batches take at most 8 results of it and every result arrives with no
`aborted`, and a source that fails after 2 results ends its stream with `aborted` after them.

In [ ]:
BOUND = {}
for form in FORMS:
    card = serve("bound-" + form)
    c = client(card, form)
    opened = [c.run(c.operation(stream_content("b%d" % n, events=-1, interval_ms=10_000))) for n in range(16)]
    over = c.run(c.operation(stream_content("b16", events=-1, interval_ms=10_000)))
    c.run(c.cancel(opened[0]["stream"]))
    again = c.run(c.operation(stream_content("b17", events=-1, interval_ms=10_000)))
    c.run(c.cancel(0))
    burst = c.run(c.operation(stream_content("burst", events=20)))
    batches = pull_until_closed(c)
    burst_stream = c.streams[burst["stream"]]
    per_batch = [sum(1 for entry in batch["entries"] if entry[0] == "next") for batch in batches]
    failing = c.run(c.operation(stream_content("fail", events=10, fail_after=2)))
    pull_until_closed(c)
    fail_stream = c.streams[failing["stream"]]
    close(c)
    served(card)
    BOUND[form] = {"opened": sum(answer["code"] == g.STREAM_OPENED for answer in opened), "17th": str(over["code"]),
                   "after_cancel": str(again["code"]), "burst_results": len(burst_stream["next"]),
                   "burst_state": burst_stream["state"], "most_in_a_batch": max(per_batch),
                   "fail_results": len(fail_stream["next"]), "fail_state": fail_stream["state"]}
    print(form, BOUND[form])

## 9. A link down ends every stream of the peer

A client opens three endless streams, takes a batch, parks a PULL and closes its socket with no
CANCEL. The card takes the link down, ends every stream of the peer and returns their sources, and
the next connection starts its count at 0. That start judges no reset, since each socket is its own
peer with a fresh endpoint and listener, which count from 0 whatever `Server::link_down` does.

So a second case takes the link down and up again on one listener. A client opens an endless
stream, takes a batch, so the count stands at 1, and parks a PULL. An OPERATION then asks the
service to take the link of its connection down and up again on the same endpoint and the same
listener while the socket stays open, as a carrier whose link drops and returns does. The listener
ends the stream and returns its source, Homa repeats the parked PULL, which is repeatable, the
listener refuses it with 404.1, and the client loses its stream and sends CANCEL of handle 0 by the
start rule. Its answer carries the count 0 only where `Server::link_down` set the count of the peer
back, and 2 where the listener kept it.

In [ ]:
LINK_DOWN = {}
for form in FORMS:
    card = serve("down-" + form, connections=2)
    c = client(card, form)
    for n in range(3):
        c.run(c.operation(stream_content("d%d" % n, events=-1, interval_ms=10_000)))
    before = c.run(c.pull())["seq"]
    park(c)
    close(c)
    time.sleep(0.2)
    second = client(card, form)
    head = second.run(second.operation(stream_content("after", events=1)))
    pull_until_closed(second)
    close(second)
    answer = served(card)
    LINK_DOWN[form] = {"seq_before": before, "returned": answer.get("service", {}).get("returned"),
                       "ended_connections": sum(conn["ended_ms"] is not None for conn in answer.get("connections", [])),
                       "first_seq_of_the_next_connection": head["seq"]}
    print(form, LINK_DOWN[form])

RELINK = {}
for form in FORMS:
    card = serve("relink-" + form)
    c = client(card, form)
    handle = c.run(c.operation(stream_content("kept", events=-1, interval_ms=10_000)))["stream"]
    before = c.run(c.pull())["seq"]
    parked = park(c)
    trigger = c.run(c.operation(g.harness_content({"run": "relink", "relink": True})))
    c.wait([parked], timeout=30)
    refused = c.take(parked)
    lost = c.streams[handle]["state"]
    start = c.start()
    close(c)
    answer = served(card)
    connections = answer.get("connections", [])
    RELINK[form] = {"seq_before": before, "trigger": trigger["answer"], "pull": str(refused["code"]),
                    "stream": lost, "start_seq": start.get("seq"), "expected": c.expected,
                    "relinks": sum(conn["relinks"] for conn in connections),
                    "restarts": sum(conn["restarts"] for conn in connections),
                    "connections": len(connections),
                    "returned": answer.get("service", {}).get("returned")}
    print(form, RELINK[form])

## Verdict

The cell lists every row with its state, raises on any row that breaks its rule, and prints the
rows this notebook does not judge yet as not run with the batch that brings them.

In [ ]:
NOT_RUN = [
    ("the client rows, the client of graphtp-homa through aosabi-link across a restart of the listener,"
     " its repeat mark, its start rule and its count of seq, gap 79 among them", "LINK in batch G6"),
    ("an introspection answer behind small operations, served by the general stack behind the listener",
     "a leftover of G3 that the record of the GraphTP track moves to G6 by default, since section 9 of"
     " docs/graphtp-in-aosabi.md gives it no exit number (gap 154)"),
    ("the 8,100 invalid documents of M2 within the page ceiling",
     "a leftover of G3 that the record of the GraphTP track moves to G6 by default, since section 9 of"
     " docs/graphtp-in-aosabi.md gives it no exit number (gap 154)"),
]
print("panics or crashes of the card:", len(PANICS))
for panic in PANICS:
    print("  ", panic)
assert not PANICS, PANICS
for form in FORMS:
    assert SIZES[form] == EXPECTED_SIZES, (form, SIZES[form])
    ten = TEN[form]
    assert ten["opened"] == 10 and ten["complete"] == 10 and ten["results"] == 30 and not ten["wrong"], ten
    assert ten["pulls"] < 40 and ten["widest_batch"] > 1 and ten["held"] == 0 and ten["dropped"] == 0, ten
    parked = PARKED[form]
    assert parked["second"] == "409.1" and parked["still_parked"] and parked["parked"] == "404.1", parked
    assert parked["pull_first"] and parked["cancel"] == ["cancelled"] and parked["returned"] == 1, parked
    reorder = REORDER[form]
    assert reorder["differing"] == 0 and reorder["lost"] == 0 and reorder["complete"] == 10, reorder
    cancels = CANCELS[form]
    assert cancels["one"]["kept_parked"] and cancels["one"]["entries"] == [("cancelled", cancels["one"]["handle"])]
    assert cancels["last"]["parked"] == "404.1" and cancels["last"]["pull_first"], cancels["last"]
    assert cancels["last"]["entries"] == [("cancelled", cancels["last"]["handle"])], cancels["last"]
    assert cancels["zero"]["parked"] == "404.1" and cancels["zero"]["pull_first"], cancels["zero"]
    assert cancels["zero"]["entries"] == sorted(("cancelled", h) for h in cancels["zero"]["handles"]), cancels
    assert cancels["returned"] == 5, cancels
    restarted = RESTARTED[form]
    assert restarted["cancel_seq"] == restarted["count_of_the_old_run"], restarted
    assert len(restarted["cancelled"]) == 2 and restarted["head_seq"] == restarted["expected_after_start"], restarted
    assert restarted["held_after_head"] == 0 and restarted["new_state"] == "complete", restarted
    assert restarted["new_results"] == [{"tick": k, "tag": "new"} for k in (1, 2, 3)], restarted
    assert restarted["dropped"] == 0 and restarted["returned"] == 2, restarted
    rows = AT_MOST_ONCE[form]
    for label, row in rows.items():
        if label.startswith("loss"):
            assert row["double_runs"] == 0 and row["other"] == 0 and row["answer_counts_wrong"] == 0, (label, row)
            assert row["unknown_marked"] == 0, (label, row)
            assert row["answered"] + row["unknown_reported"] == row["operations"], (label, row)
    unmarked, marked = rows["restart after the run, unmarked"], rows["restart after the run, marked"]
    assert unmarked["answer"] == "unknown" and unmarked["code"] == "502.1" and unmarked["runs"] == 1, unmarked
    assert marked["answer"] == "result" and marked["runs"] == 2 and marked["count_in_answer"] == 2, marked
    assert unmarked["restarts"] == 1 and marked["restarts"] == 1, (unmarked, marked)
    unmarked, marked = rows["restart before the run, unmarked"], rows["restart before the run, marked"]
    assert unmarked["answer"] == "unknown" and unmarked["code"] == "502.1" and unmarked["runs"] == 0, unmarked
    assert marked["answer"] == "result" and marked["runs"] == 1 and marked["count_in_answer"] == 1, marked
    assert unmarked["restarts"] == 1 and marked["restarts"] == 1, (unmarked, marked)
    assert unmarked["trigger"] == "result" and marked["trigger"] == "result", (unmarked, marked)
    assert rows["link down"]["answer"] == "unknown" and rows["link down"]["code"] == "502.2", rows["link down"]
    assert rows["link down"]["runs"] == 1, rows["link down"]
    silent = rows["silent past the dead peer rule"]
    assert silent["answer"] == "unknown" and silent["runs"] == 1 and silent["peer_deaths"] >= 1, silent
    across = rows["parked PULL across a restart"]
    assert across["pull"] == "404.1" and across["stream"] == "lost" and across["start_seq"] == 0, across
    assert across["expected"] == 1 and across["restarts"] == 1, across
    bound = BOUND[form]
    assert bound["opened"] == 16 and bound["17th"] == "429.1" and bound["after_cancel"] == "200.3", bound
    assert bound["burst_results"] == 20 and bound["burst_state"] == "complete" and bound["most_in_a_batch"] <= 8, bound
    assert bound["fail_results"] == 2 and bound["fail_state"] == "aborted", bound
    down = LINK_DOWN[form]
    assert down["returned"] == 3 and down["ended_connections"] == 2 and down["first_seq_of_the_next_connection"] == 0, down
    relink = RELINK[form]
    assert relink["seq_before"] >= 1 and relink["trigger"] == "result" and relink["pull"] == "404.1", relink
    assert relink["stream"] == "lost" and relink["start_seq"] == 0 and relink["expected"] == 1, relink
    assert relink["relinks"] == 1 and relink["restarts"] == 0 and relink["connections"] == 1, relink
    assert relink["returned"] == 1, relink
print()
print("measured on the forms %s:" % ", ".join(FORMS))
for form in FORMS:
    loss = {label: (row["operations"], row["answered"], row["unknown_reported"], row["double_runs"])
            for label, row in AT_MOST_ONCE[form].items() if label.startswith("loss")}
    print("   %s: the sizes of section 6.2 %s" % (form, SIZES[form]))
    print("   %s: ten streams over %d PULLs, at most %d streams in a batch" % (form, TEN[form]["pulls"],
                                                                                TEN[form]["widest_batch"]))
    print("   %s: %d stream answers applied in %d orders, 0 differing" % (form, REORDER[form]["answers"],
                                                                         permutations))
    print("   %s: loss runs (operations, answered, unknown reported, double runs) %s" % (form, loss))
print()
print("not run in this notebook:")
for part, batch in NOT_RUN:
    print("   %s: not run, %s" % (part, batch))
print()
print("VERDICT: on %s, the sizes of section 6.2 held, one parked PULL served ten streams, a second PULL"
      " answered 409.1, the stream answers applied by seq in %d orders with 0 lost events, CANCEL of the"
      " peer's only handle, of its last handle and of handle 0 each answered the parked PULL with 404.1"
      " first while CANCEL of one handle of two left it parked, a restarted client took its count from"
      " CANCEL of handle 0, the restart cases ran with 0 double runs of an unmarked OPERATION and every"
      " unknown outcome reported, 429.1 answered at the bound, a link down ended every stream of the peer"
      " and set its count back to 0 on the same listener, and the client rows not run"
      % (" and ".join(FORMS), permutations))